<a href="https://colab.research.google.com/github/VidhitaYadav/Chest-XRay-Classification-ResNet50/blob/main/MACXRay2_with_Sobel_Github.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input
import numpy as np
import matplotlib.pyplot as plt


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
data_dir="/content/drive/MyDrive/MultiAbnormalityCXR"
train=data_dir+"/train"
test=data_dir+"/test"


In [ ]:
import tensorflow as tf
train_dataset=tf.keras.preprocessing.image_dataset_from_directory(
    train,
    validation_split=0.2,
    subset="training",
    seed=42,
    image_size=(224,224),
    batch_size=64
    )

In [ ]:
import tensorflow as tf
val_dataset=tf.keras.preprocessing.image_dataset_from_directory(
    train,
    validation_split=0.2,
    subset="validation",
    seed=42,
    image_size=(224,224),
    batch_size=64
    )

In [ ]:
import tensorflow as tf
test_dataset=tf.keras.preprocessing.image_dataset_from_directory(
    test,
    image_size=(224*224),
    batch_size=64,
    shuffle=True
    )

class_names=train_dataset.class_names
print("Classes:",class_names)

In [ ]:
for images,labels in train_dataset.take(1):
  print("Single image shape:",images[0].shape)
  print()


In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(10,10))
for images ,labels in train_dataset.take(1):
  for i in range(9):
    x=plt.subplot(3,3,i+1)
    plt.imshow(images[i].numpy().astype("uint8"))
    plt.title(class_names[labels[i]])
    plt.axis("off")

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

In [ ]:
def preprocess_resnet(image, label):
    image = tf.cast(image, tf.float32)
    image = preprocess_input(image)
    return image, label

train_dataset = train_dataset.map(preprocess_resnet, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)
val_dataset   = val_dataset.map(preprocess_resnet, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)
test_dataset  = test_dataset.map(preprocess_resnet, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)


In [ ]:
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.1),
    layers.RandomContrast(0.1)
])


In [ ]:
from tensorflow.keras.applications import ResNet50
from tensorflow.keras import layers, models


In [ ]:
INPUT_SHAPE = (224, 224, 3)

base_model = ResNet50(
    weights="imagenet",
    include_top=False,
    input_shape=INPUT_SHAPE
)

base_model.trainable = False


In [ ]:
model = models.Sequential([

    # ResNet50 backbone
    base_model,

    # ---------- CNN Refinement Block ----------
    layers.Conv2D(256, (3,3), padding="same", activation="relu"),
    layers.BatchNormalization(),

    layers.Conv2D(128, (3,3), padding="same", activation="relu"),
    layers.BatchNormalization(),

    layers.Conv2D(64, (3,3), padding="same", activation="relu"),
    layers.BatchNormalization(),

    layers.Conv2D(32, (3,3), padding="same", activation="relu"),
    # ------------------------------------------

    layers.GlobalAveragePooling2D(),

    layers.Dense(128, activation="relu"),
    layers.Dropout(0.5),

    layers.Dense(len(class_names), activation="softmax")
])


In [ ]:

tf.keras.backend.clear_session()


In [ ]:
target_layer = base_model.get_layer("conv4_block6_out")


In [ ]:
img_path = "/content/drive/MyDrive/MultiAbnormalityCXR/test/Pneumonia/person1_virus_6.jpeg"  # change path

img = keras.utils.load_img(
    img_path,
    target_size=(224, 224),
    color_mode="rgb"
)

img_array = keras.utils.img_to_array(img)
img_array = img_array / 255.0
img_array = np.expand_dims(img_array, axis=0)


In [ ]:
from tensorflow import keras
from tensorflow.keras import layers

# Collect all Conv2D layers
conv_layers = [layer for layer in base_model.layers if isinstance(layer, layers.Conv2D)]

print("Total Conv Layers:", len(conv_layers))

# Safety check
if len(conv_layers) < 2:
    raise ValueError("Not enough Conv2D layers found in base_model")

# Select early convolution layer
target_layer = conv_layers[1]   # conv2_block1_1_conv
print("Target Layer:", target_layer.name)


In [ ]:
target_layer = base_model.get_layer("conv2_block1_1_conv")

In [ ]:
feature_model = keras.Model(
    inputs=base_model.input,
    outputs=target_layer.output
)


In [ ]:
feature_maps = feature_model(img_array, training=False).numpy()


In [ ]:
plt.figure(figsize=(12,4))

for i in range(6):
    plt.subplot(1,6,i+1)
    plt.imshow(feature_maps[0, :, :, i], cmap="gray")
    plt.axis("off")
    plt.title(f"Map {i+1}")

plt.suptitle("Deep Feature Maps from ResNet50 (conv4_block6_out)")
plt.show()


In [ ]:
plt.figure(figsize=(14,4))

for i in range(6):
    plt.subplot(1,6,i+1)
    plt.imshow(
        feature_maps[0, :, :, i],
        cmap="gray",
        interpolation="nearest"
    )
    plt.axis("off")
    plt.title(f"Feature {i+1}")

plt.suptitle("Intermediate Feature Maps from ResNet50 (conv4_block6_out)", fontsize=12)
plt.tight_layout()
plt.show()


In [ ]:
from tensorflow import keras
from tensorflow.keras import layers

# Collect all Conv2D layers
conv_layers = [layer for layer in base_model.layers if isinstance(layer, layers.Conv2D)]

print("Total Conv Layers:", len(conv_layers))

# Safety check
if len(conv_layers) < 2:
    raise ValueError("Not enough Conv2D layers found in base_model")

# Select early convolution layer
target_layer = conv_layers[1]   # conv2_block1_1_conv
print("Target Layer:", target_layer.name)


In [ ]:
target_layer = conv_layers[1]

layer_model = keras.Model(
    inputs=base_model.input,
    outputs=target_layer.output
)


In [ ]:
plt.imshow(feature_maps[0, :, :, i], cmap="gray")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.figure(figsize=(14,4))

for i in range(6):
    fmap = feature_maps[0, :, :, i]

    # Normalize feature map (for visualization only)
    fmap = (fmap - np.min(fmap)) / (np.max(fmap) - np.min(fmap) + 1e-8)

    plt.subplot(1,6,i+1)
    plt.imshow(fmap, cmap="gray", interpolation="bilinear")
    plt.axis("off")
    plt.title(f"Map {i+1}")

plt.suptitle("Feature Maps from ResNet50 ")
plt.show()


In [ ]:
from tensorflow import keras
from tensorflow.keras import layers

conv_layers = [l for l in base_model.layers if isinstance(l, layers.Conv2D)]
target_layer = conv_layers[-3]   # deeper layer → better semantics

feature_model = keras.Model(
    inputs=base_model.input,
    outputs=target_layer.output
)

print("Using layer:", target_layer.name)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.preprocessing import image
from tensorflow import keras

target_layer = base_model.get_layer("conv1_conv")

feature_model = keras.Model(
    inputs=base_model.input,
    outputs=target_layer.output
)


In [ ]:
img_path = "/content/drive/MyDrive/MultiAbnormalityCXR/test/Atelectasis/Atelectasis (13).jpg"

img = image.load_img(img_path, target_size=(224,224), color_mode="rgb")
img_array = image.img_to_array(img) / 255.0
img_array = np.expand_dims(img_array, axis=0)

feature_maps = feature_model.predict(img_array, verbose=0)

plt.figure(figsize=(14,4))

plt.subplot(1,7,1)
plt.imshow(img_array[0])
plt.title("Input X-ray")
plt.axis("off")

for i in range(6):
    fmap = feature_maps[0,:,:,i]
    fmap = (fmap - fmap.min()) / (fmap.max() - fmap.min() + 1e-8)
    plt.subplot(1,7,i+2)
    plt.imshow(fmap, cmap="gray")
    plt.axis("off")
    plt.title(f"Map {i+1}")

plt.suptitle("Feature Map Visualization – Atelectasis")
plt.show()


In [ ]:
img_path = "/content/drive/MyDrive/MultiAbnormalityCXR/test/Cardiomegaly/28.png"

img = image.load_img(img_path, target_size=(224,224))
img_array = image.img_to_array(img) / 255.0
img_array = np.expand_dims(img_array, axis=0)

feature_maps = feature_model.predict(img_array, verbose=0)

plt.figure(figsize=(14,4))

plt.subplot(1,7,1)
plt.imshow(img_array[0])
plt.axis("off")
plt.title("Input X-ray")

for i in range(6):
    fmap = feature_maps[0,:,:,i]
    fmap = (fmap - fmap.min()) / (fmap.max() - fmap.min() + 1e-8)
    plt.subplot(1,7,i+2)
    plt.imshow(fmap, cmap="gray")
    plt.axis("off")
    plt.title(f"Map {i+1}")

plt.suptitle("Feature Map Visualization – Cardiomegaly")
plt.show()


In [ ]:
img_path = "/content/drive/MyDrive/MultiAbnormalityCXR/test/Emphysema/00000005_003.png"

img = image.load_img(img_path, target_size=(224,224))
img_array = image.img_to_array(img) / 255.0
img_array = np.expand_dims(img_array, axis=0)

feature_maps = feature_model.predict(img_array, verbose=0)

plt.figure(figsize=(14,4))

plt.subplot(1,7,1)
plt.imshow(img_array[0])
plt.axis("off")
plt.title("Input X-ray")

for i in range(6):
    fmap = feature_maps[0,:,:,i]
    fmap = (fmap - fmap.min()) / (fmap.max() - fmap.min() + 1e-8)
    plt.subplot(1,7,i+2)
    plt.imshow(fmap, cmap="gray")
    plt.axis("off")
    plt.title(f"Map {i+1}")

plt.suptitle("Feature Map Visualization – Emphysema")
plt.show()


In [ ]:
img_path = "/content/drive/MyDrive/MultiAbnormalityCXR/test/Lung Hernia/generated_image_hernia62.png"  # <-- Please update this with an actual existing filename from your drive, e.g., run `!ls /content/drive/MyDrive/MultiAbnormalityCXR/test/Hernia/` to see available files.

img = image.load_img(img_path, target_size=(224,224))
img_array = image.img_to_array(img) / 255.0
img_array = np.expand_dims(img_array, axis=0)

feature_maps = feature_model.predict(img_array, verbose=0)

plt.figure(figsize=(14,4))

plt.subplot(1,7,1)
plt.imshow(img_array[0])
plt.axis("off")
plt.title("Input X-ray")

for i in range(6):
    fmap = feature_maps[0,:,:,i]
    fmap = (fmap - fmap.min()) / (fmap.max() - fmap.min() + 1e-8)
    plt.subplot(1,7,i+2)
    plt.imshow(fmap, cmap="gray")
    plt.axis("off")
    plt.title(f"Map {i+1}")

plt.suptitle("Feature Map Visualization – Lung Hernia")
plt.show()

In [ ]:
img_path = "/content/drive/MyDrive/MultiAbnormalityCXR/test/Normal/IM-0007-0001.jpeg"

img = image.load_img(img_path, target_size=(224,224))
img_array = image.img_to_array(img) / 255.0
img_array = np.expand_dims(img_array, axis=0)

feature_maps = feature_model.predict(img_array, verbose=0)

plt.figure(figsize=(14,4))

plt.subplot(1,7,1)
plt.imshow(img_array[0])
plt.axis("off")
plt.title("Input X-ray")

for i in range(6):
    fmap = feature_maps[0,:,:,i]
    fmap = (fmap - fmap.min()) / (fmap.max() - fmap.min() + 1e-8)
    plt.subplot(1,7,i+2)
    plt.imshow(fmap, cmap="gray")
    plt.axis("off")
    plt.title(f"Map {i+1}")

plt.suptitle("Feature Map Visualization – Normal")
plt.show()


In [ ]:
img_path = "/content/drive/MyDrive/MultiAbnormalityCXR/test/Pneumonia/person1_virus_6.jpeg"

img = image.load_img(img_path, target_size=(224,224))
img_array = image.img_to_array(img) / 255.0
img_array = np.expand_dims(img_array, axis=0)

feature_maps = feature_model.predict(img_array, verbose=0)

plt.figure(figsize=(14,4))

plt.subplot(1,7,1)
plt.imshow(img_array[0])
plt.axis("off")
plt.title("Input X-ray")

for i in range(6):
    fmap = feature_maps[0,:,:,i]
    fmap = (fmap - fmap.min()) / (fmap.max() - fmap.min() + 1e-8)
    plt.subplot(1,7,i+2)
    plt.imshow(fmap, cmap="gray")
    plt.axis("off")
    plt.title(f"Map {i+1}")

plt.suptitle("Feature Map Visualization – Pneumonia")
plt.show()


In [ ]:
img_path = "/content/drive/MyDrive/MultiAbnormalityCXR/test/Tuberculosis/Tuberculosis-48.png"

img = image.load_img(img_path, target_size=(224,224))
img_array = image.img_to_array(img) / 255.0
img_array = np.expand_dims(img_array, axis=0)

feature_maps = feature_model.predict(img_array, verbose=0)

plt.figure(figsize=(14,4))

plt.subplot(1,7,1)
plt.imshow(img_array[0])
plt.axis("off")
plt.title("Input X-ray")

for i in range(6):
    fmap = feature_maps[0,:,:,i]
    fmap = (fmap - fmap.min()) / (fmap.max() - fmap.min() + 1e-8)
    plt.subplot(1,7,i+2)
    plt.imshow(fmap, cmap="gray")
    plt.axis("off")
    plt.title(f"Map {i+1}")

plt.suptitle("Feature Map Visualization – Tuberculosis")
plt.show()


In [ ]:
import tensorflow as tf
from tensorflow.keras import layers
import matplotlib.pyplot as plt
import numpy as np

# ---------- LOAD ONE CHEST X-RAY IMAGE ----------
img_path = "/content/drive/MyDrive/MultiAbnormalityCXR/test/Atelectasis/Atelectasis (70).jpg"   # <-- CHANGE THIS PATH

sample_img = tf.keras.preprocessing.image.load_img(
    img_path, target_size=(224, 224), color_mode="rgb"
)
sample_img = tf.keras.preprocessing.image.img_to_array(sample_img)
sample_img = sample_img / 255.0
sample_img = tf.expand_dims(sample_img, axis=0)  # shape: (1,224,224,3)

# ---------- RESIZE & CONVERT TO GRAYSCALE ----------
img = tf.image.resize(sample_img, (32, 32))
img = tf.image.rgb_to_grayscale(img)
# ---------- DISPLAY INPUT IMAGE ----------
plt.figure(figsize=(4,4))
plt.imshow(tf.squeeze(img), cmap="gray")
plt.title("Input Chest X-ray (32×32)")
plt.axis("off")
plt.show()





In [ ]:
# ---------- WITHOUT PADDING ----------
no_pad = layers.Conv2D(
    filters=1,
    kernel_size=3,
    padding='valid',
    kernel_initializer='ones',
    use_bias=False
)(img)

plt.figure(figsize=(4,4))
plt.imshow(tf.squeeze(no_pad), cmap="gray")
plt.title("Without Padding (Valid)")
plt.axis("off")
plt.show()

In [ ]:
# ---------- WITH PADDING ----------
pad = layers.Conv2D(
    filters=1,
    kernel_size=3,
    padding='same',
    kernel_initializer='ones',
    use_bias=False
)(img)

plt.figure(figsize=(4,4))
plt.imshow(tf.squeeze(pad), cmap="gray")
plt.title("With Padding (Same)")
plt.axis("off")
plt.show()

In [ ]:
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow.keras import layers
import numpy as np
from PIL import Image

# --------------------------------------------------
# 1. Load a real chest X-ray image
# --------------------------------------------------
img_path = "/content/drive/MyDrive/MultiAbnormalityCXR/test/Atelectasis/Atelectasis (70).jpg"  # CHANGE to your image path

img = Image.open(img_path).convert("L")   # grayscale
img = img.resize((128, 128))              # moderate resolution

img_arr = np.array(img).astype("float32")
img_arr = img_arr / 255.0                 # normalize
img_arr = img_arr[np.newaxis, ..., np.newaxis]  # (1,128,128,1)

# --------------------------------------------------
# 2. Display original image (clear visualization)
# --------------------------------------------------
plt.figure(figsize=(4,4))
plt.imshow(img_arr[0,:,:,0], cmap="gray", interpolation="bilinear")
plt.title("Original Chest X-ray")
plt.axis("off")
plt.show()





In [ ]:
# --------------------------------------------------
# 3. Convolution WITHOUT padding (VALID)
# --------------------------------------------------
no_pad = layers.Conv2D(
    filters=1,
    kernel_size=3,
    padding="valid",
    kernel_initializer="ones",
    use_bias=False
)(img_arr)

plt.figure(figsize=(4,4))
plt.imshow(no_pad[0,:,:,0], cmap="gray", interpolation="bilinear")
plt.title("Without Padding (VALID)")
plt.axis("off")
plt.show()

In [ ]:
# --------------------------------------------------
# 4. Convolution WITH padding (SAME)
# --------------------------------------------------
pad = layers.Conv2D(
    filters=1,
    kernel_size=3,
    padding="same",
    kernel_initializer="ones",
    use_bias=False
)(img_arr)

plt.figure(figsize=(4,4))
plt.imshow(pad[0,:,:,0], cmap="gray", interpolation="bilinear")
plt.title("With Padding (SAME)")
plt.axis("off")
plt.show()


In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras import layers
from PIL import Image

# ---------- Load and preprocess image ----------
img_path = "/content/drive/MyDrive/MultiAbnormalityCXR/test/Atelectasis/Atelectasis (70).jpg"   # change if needed

img = Image.open(img_path).convert("L")     # grayscale
img = img.resize((224, 224))                # HIGH clarity (ResNet50 compatible)

img_arr = np.array(img).astype("float32") / 255.0
img_arr = img_arr[np.newaxis, ..., np.newaxis]  # (1,224,224,1)

# ---------- Display original image ----------
plt.figure(figsize=(5,5))
plt.imshow(img_arr[0,:,:,0], cmap="gray", interpolation="bilinear")
plt.title("Input Chest X-ray (224×224)")
plt.axis("off")
plt.show()




In [ ]:
# ---------- VALID padding ----------
conv_valid = layers.Conv2D(
    filters=1,
    kernel_size=3,
    padding='valid',
    kernel_initializer='ones'
)(img_arr)

plt.figure(figsize=(5,5))
plt.imshow(conv_valid[0,:,:,0], cmap="gray", interpolation="bilinear")
plt.title("Convolution Output (VALID Padding)")
plt.axis("off")
plt.show()


In [ ]:
# ---------- SAME padding ----------
conv_same = layers.Conv2D(
    filters=1,
    kernel_size=3,
    padding='same',
    kernel_initializer='ones'
)(img_arr)

plt.figure(figsize=(5,5))
plt.imshow(conv_same[0,:,:,0], cmap="gray", interpolation="bilinear")
plt.title("Convolution Output (SAME Padding)")
plt.axis("off")
plt.show()


In [ ]:
squeezed_no_pad=tf.squeeze(no_pad)
print(squeezed_no_pad.shape)


In [ ]:
no_pad.shape

In [ ]:
squeezed_pad=tf.squeeze(pad)
print(squeezed_pad.shape)


In [ ]:
pad.shape

In [ ]:
from google.colab import files
from PIL import Image
uploaded=files.upload()
filename=list(uploaded.keys())[0]
image=Image.open(filename).convert('L').resize((128,128))
plt.imshow(image,cmap='gray')
plt.title("Input Image")
plt.axis("off")
plt.show()

In [ ]:
img_arr=np.array(image)
img_arr=img_arr/255.0
img_arr=img_arr.reshape((1,128,128,1)).astype('float32')

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np


In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np
import keras.ops as ops


# =====================================================
# Sobel Edge Extraction Layer
# =====================================================
class SobelEdgeLayer(layers.Layer):
    def __init__(self, **kwargs):
        super().__init__(trainable=False, **kwargs)

        # Sobel kernels
        sobel_x = np.array([[1, 0, -1],
                            [2, 0, -2],
                            [1, 0, -1]], dtype=np.float32).reshape(3, 3, 1, 1)

        sobel_y = np.array([[1,  2,  1],
                            [0,  0,  0],
                            [-1, -2, -1]], dtype=np.float32).reshape(3, 3, 1, 1)

        # RGB → Grayscale (1x1 convolution)
        grayscale_weights = np.array(
            [[[[0.2989]], [[0.5870]], [[0.1140]]]], dtype=np.float32
        )

        self.gray_conv = layers.Conv2D(
            1, (1, 1), padding="same", use_bias=False,
            trainable=False,
            kernel_initializer=tf.constant_initializer(grayscale_weights)
        )

        self.sobel_x = layers.Conv2D(
            1, (3, 3), padding="same", use_bias=False,
            trainable=False,
            kernel_initializer=tf.constant_initializer(sobel_x)
        )

        self.sobel_y = layers.Conv2D(
            1, (3, 3), padding="same", use_bias=False,
            trainable=False,
            kernel_initializer=tf.constant_initializer(sobel_y)
        )

    def call(self, inputs):
        gray = self.gray_conv(inputs)
        gx = self.sobel_x(gray)
        gy = self.sobel_y(gray)

        edges = ops.sqrt(ops.square(gx) + ops.square(gy))
        edges = edges / (ops.max(edges) + 1e-6)

        return edges


# =====================================================
# Sobel Enhancement Layer (Concatenation)
# =====================================================
class SobelEnhancementLayer(layers.Layer):
    def __init__(self, **kwargs):
        super().__init__(trainable=False, **kwargs)
        self.sobel = SobelEdgeLayer()
        self.concat = layers.Concatenate(axis=-1)

    def call(self, inputs):
        edges = self.sobel(inputs)
        return self.concat([inputs, edges, edges, edges])


In [ ]:
inputs = keras.Input(shape=(224, 224, 3))

# Sobel Edge Enhancement
x = SobelEnhancementLayer()(inputs)   # (224, 224, 6)

# Project back to 3 channels for ResNet50
x = layers.Conv2D(3, (1, 1), padding="same", trainable=False)(x)


In [ ]:
import keras.ops as ops # Ensure keras.ops is imported here as well

def sobel_preprocess(inputs):
    sobel_layer = SobelEdgeLayer()
    edges = sobel_layer(inputs)
    return tf.concat([inputs, edges, edges, edges], axis=-1)


In [ ]:
inputs = keras.Input(shape=(224, 224, 3))

x = SobelEnhancementLayer()(inputs)
x = layers.Conv2D(3, (1, 1), padding="same", trainable=False)(x)

base_model = ResNet50(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)
base_model.trainable = False

x = base_model(x)


In [ ]:
x = layers.GlobalAveragePooling2D()(x)
x = layers.BatchNormalization()(x)
x = layers.Dense(256, activation="relu")(x)
x = layers.Dropout(0.5)(x)

outputs = layers.Dense(7, activation="softmax")(x)

model = keras.Model(inputs, outputs)

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(1e-4),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)


In [ ]:
from tensorflow.keras.callbacks import ModelCheckpoint, CSVLogger

# Google Drive paths
BEST_MODEL_PATH = "/content/drive/MyDrive/sobel_resnet50_best.keras"
LAST_MODEL_PATH = "/content/drive/MyDrive/sobel_resnet50_last.keras"
HISTORY_PATH    = "/content/drive/MyDrive/sobel_training_log.csv"


In [ ]:
# Save best model based on validation loss
checkpoint_best = ModelCheckpoint(
    filepath=BEST_MODEL_PATH,
    monitor="val_loss",
    save_best_only=True,
    verbose=1
)

# Save model after every epoch (latest state)
checkpoint_last = ModelCheckpoint(
    filepath=LAST_MODEL_PATH,
    save_best_only=False,
    verbose=0
)

# Save training history to CSV (safe backup)
csv_logger = CSVLogger(HISTORY_PATH, append=True)


In [ ]:
history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=50,
    callbacks=[
        checkpoint_best,
        checkpoint_last,
        csv_logger
    ]
)

In [ ]:
import os

print("Best model exists:", os.path.exists(BEST_MODEL_PATH))
print("Last model exists:", os.path.exists(LAST_MODEL_PATH))
print("History log exists:", os.path.exists(HISTORY_PATH))


In [ ]:

import tensorflow as tf

gpus = tf.config.list_physical_devices('GPU')
print(gpus)



In [ ]:
MODEL_PATH = "/content/drive/MyDrive/chest_xray_resnet50_latest.keras"
BEST_MODEL_PATH = "/content/drive/MyDrive/chest_xray_resnet50_best.keras"


In [ ]:
import tensorflow as tf
import numpy as np
import cv2
import matplotlib.pyplot as plt


In [ ]:
def make_gradcam_heatmap(img_array, model, last_conv_layer_name, pred_index=None):
    grad_model = tf.keras.models.Model(
        [model.inputs],
        [model.get_layer(last_conv_layer_name).output, model.output]
    )

    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_array)
        if pred_index is None:
            pred_index = tf.argmax(predictions[0])
        class_channel = predictions[:, pred_index]

    grads = tape.gradient(class_channel, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

    conv_outputs = conv_outputs[0]
    heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)

    heatmap = tf.maximum(heatmap, 0)
    heatmap /= tf.reduce_max(heatmap) + 1e-8

    return heatmap.numpy()


In [ ]:
img_path = "/content/drive/MyDrive/MultiAbnormalityCXR/test/Atelectasis/Atelectasis (70).jpg"

img = tf.keras.preprocessing.image.load_img(img_path, target_size=(224,224))
img_array = tf.keras.preprocessing.image.img_to_array(img)
img_array = np.expand_dims(img_array, axis=0)
img_array = preprocess_input(img_array)

In [ ]:
resnet = model.get_layer("resnet50")


In [ ]:
print([l.name for l in resnet.layers if "conv5" in l.name])


In [ ]:
last_conv_layer_name = "conv5_block3_out"


In [ ]:
from tensorflow import keras
model = keras.models.load_model(BEST_MODEL_PATH)

In [ ]:
model.save("/content/chest_xray_resnet50_best.keras")


In [ ]:
model.save("/content/chest_xray_resnet50_best.h5")


In [ ]:
model.save("/content/drive/MyDrive/chest_xray_resnet50_best.keras")


In [ ]:
import tensorflow as tf

test_dir = "/content/drive/MyDrive/MultiAbnormalityCXR/test"

class_names = [
    'Atelectasis',
    'Cardiomegaly',
    'Emphysema',
    'Lung Hernia',
    'Normal',
    'Pneumonia',
    'Tuberculosis'
]


In [ ]:
from tensorflow.keras.applications.resnet50 import preprocess_input

test_dataset = tf.keras.preprocessing.image_dataset_from_directory(
    test_dir,
    image_size=(224, 224),
    batch_size=32,
    shuffle=False,
    label_mode='int'
)
print(test_dataset.class_names)

test_dataset = test_dataset.map(lambda x, y: (preprocess_input(x), y)).prefetch(AUTOTUNE)

In [ ]:
import os

print("Test directory exists:", os.path.exists(test_dir))
print("Subfolders:", os.listdir(test_dir))


In [ ]:
for images, labels in test_dataset.take(1):
    print("Images shape:", images.shape)
    print("Labels shape:", labels.shape)


In [ ]:
import tensorflow as tf
from tensorflow.keras.applications.resnet50 import preprocess_input

test_dataset = tf.keras.utils.image_dataset_from_directory(
    test_dir,
    image_size=(224, 224),
    batch_size=32,
    shuffle=False
)

test_dataset = test_dataset.map(
    lambda x, y: (preprocess_input(x), y),
    num_parallel_calls=tf.data.AUTOTUNE
)

test_dataset = test_dataset.prefetch(tf.data.AUTOTUNE)


In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score

y_true = []
y_pred = []

for images, labels in test_dataset.as_numpy_iterator():
    preds = model.predict(images, verbose=0)
    y_pred.extend(np.argmax(preds, axis=1))
    y_true.extend(labels)

print("Test Accuracy:", accuracy_score(y_true, y_pred) * 100)


In [ ]:
model.summary()

In [ ]:
len(model.layers)

In [ ]:
import plotly.express as px

acc_data = {
    "Train Accuracy": history.history['accuracy'],
    "Val Accuracy": history.history['val_accuracy']
}

fig = px.line(acc_data,
              labels={"value":"Accuracy","index":"Epochs"},
              title="Accuracy Curve")
fig.show()

In [ ]:
import plotly.express as px

acc_data = {
    "Train Loss": history.history['loss'],
    "Val Loss": history.history['val_loss']
}

fig = px.line(acc_data,
              labels={"value":"Loss","index":"Epochs"},
              title="Loss Curve")
fig.show()

In [ ]:
test_dataset=tf.keras.preprocessing.image_dataset_from_directory(
    test,
    image_size=(224,224),#width X height
    batch_size=64,  #32batches
    shuffle=True, # Set shuffle to True
    color_mode='rgb',
    label_mode='int',
    class_names=class_names # Include all class names
)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.plot(history.history['accuracy'],label="train accuracy")
plt.plot(history.history['val_accuracy'],label="val accuracy")
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.title('Accuracy curve')
plt.xticks(np.arange(1,21,1))
plt.show()

In [ ]:
import matplotlib.pyplot as plt
plt.plot(history.history['loss'],label="train loss")
plt.plot(history.history['val_loss'],label="val loss")
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.title('Loss curve')
plt.xticks(np.arange(1,21,1))
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix
import seaborn as sns

y_pred = model.predict(test_dataset)
y_pred_classes = np.argmax(y_pred, axis=1)


y_true = np.concatenate([y.numpy() for x, y in test_dataset], axis=0)


cm = confusion_matrix(y_true, y_pred_classes)


plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names,
            yticklabels=class_names)
plt.title("Confusion Matrix")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.show()


In [ ]:
from sklearn.metrics import classification_report, accuracy_score

print("\nClassification Report:")
print(classification_report(y_true, y_pred_classes, target_names=class_names))

print("\nAccuracy:", accuracy_score(y_true, y_pred_classes))


In [ ]:
import numpy as np
from sklearn.metrics import classification_report, accuracy_score

# ---------- TRAIN SET ----------
y_train_true = []
y_train_pred = []

for images, labels in train_dataset:
    preds = model.predict(images, verbose=0)
    y_train_pred.extend(np.argmax(preds, axis=1))
    y_train_true.extend(labels.numpy())

y_train_true = np.array(y_train_true)
y_train_pred = np.array(y_train_pred)


In [ ]:
print("\n================ TRAIN CLASSIFICATION REPORT ================\n")
print(classification_report(
    y_train_true,
    y_train_pred,
    target_names=class_names,
    digits=4
))

print("Train Accuracy:", accuracy_score(y_train_true, y_train_pred))





In [ ]:
print("\n================ TEST CLASSIFICATION REPORT ================\n")
print(classification_report(
    y_true,
    y_pred,
    target_names=class_names,
    digits=4
))

print("Test Accuracy:", accuracy_score(y_true, y_pred))

In [ ]:
y_pred_classes

In [ ]:
y_true

In [ ]:
y_true.shape

In [ ]:
for images, labels in test_dataset.take(1):

    Unseen_sample = images[8].numpy()
    actual_label  = labels[8].numpy()

    y_prob = model.predict(tf.expand_dims(Unseen_sample, axis=0))
    y_pred = np.argmax(y_prob)

    print("Actual Label:", class_names[actual_label])
    print("Predicted Label:", class_names[y_pred])

    plt.imshow(Unseen_sample.astype("uint8"))
    plt.title(f"Actual: {class_names[actual_label]} | Predicted: {class_names[y_pred]}")
    plt.axis("off")
    plt.show()
    break

In [ ]:
for images, labels in test_dataset.take(1):

    Unseen_sample = images[9].numpy()
    actual_label  = labels[9].numpy()

    y_prob = model.predict(tf.expand_dims(Unseen_sample, axis=0))
    y_pred = np.argmax(y_prob)

    print("Actual Label:", class_names[actual_label])
    print("Predicted Label:", class_names[y_pred])

    plt.imshow(Unseen_sample.astype("uint8"))
    plt.title(f"Actual: {class_names[actual_label]} | Predicted: {class_names[y_pred]}")
    plt.axis("off")
    plt.show()
    break

In [ ]:
for images, labels in test_dataset.take(5):

    Unseen_sample = images[1].numpy()
    actual_label  = labels[1].numpy()

    y_prob = model.predict(tf.expand_dims(Unseen_sample, axis=0))
    y_pred = np.argmax(y_prob)

    print("Actual Label:", class_names[actual_label])
    print("Predicted Label:", class_names[y_pred])

    plt.imshow(Unseen_sample.astype("uint8"))
    plt.title(f"Actual: {class_names[actual_label]} | Predicted: {class_names[y_pred]}")
    plt.axis("off")
    plt.show()
    break

In [ ]:
for images, labels in test_dataset.take(1):

    Unseen_sample = images[13].numpy()
    actual_label  = labels[13].numpy()

    y_prob = model.predict(tf.expand_dims(Unseen_sample, axis=0))
    y_pred = np.argmax(y_prob)

    print("Actual Label:", class_names[actual_label])
    print("Predicted Label:", class_names[y_pred])

    plt.imshow(Unseen_sample.astype("uint8"))
    plt.title(f"Actual: {class_names[actual_label]} | Predicted: {class_names[y_pred]}")
    plt.axis("off")
    plt.show()
    break

In [ ]:
for images, labels in test_dataset.take(1):
    print("Actual labels in the first batch:", [class_names[label.numpy()] for label in labels])
    break

In [ ]:
for images, labels in test_dataset.skip(1).take(1):
    print("Actual labels in the second batch:", [class_names[label.numpy()] for label in labels])

    # Display predictions for the first few images in this batch
    plt.figure(figsize=(10, 10))
    for i in range(min(9, images.shape[0])): # Display up to 9 images
        Unseen_sample = images[i].numpy()
        actual_label  = labels[i].numpy()

        y_prob = model.predict(tf.expand_dims(Unseen_sample, axis=0))
        y_pred = np.argmax(y_prob)

        plt.subplot(3, 3, i + 1)
        plt.imshow(Unseen_sample.astype("uint8"))
        plt.title(f"Actual: {class_names[actual_label]} | Predicted: {class_names[y_pred]}")
        plt.axis("off")
    plt.tight_layout()
    plt.subplots_adjust(hspace=1.0) # Increase hspace further
    plt.show()
    break

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# Assuming cm (confusion matrix) and class_names are already defined from previous cells

sensitivity = dict()
specificity = dict()
for i in range(len(class_names)):
    # True Positives for class i
    TP = cm[i, i]
    # False Negatives for class i
    FN = np.sum(cm[i, :]) - TP
    # False Positives for class i
    FP = np.sum(cm[:, i]) - TP
    # True Negatives for class i (total samples - TP - FN - FP)
    TN = np.sum(cm) - TP - FN - FP

    # Sensitivity (True Positive Rate) = TP / (TP + FN)
    sensitivity[class_names[i]] = TP / (TP + FN) if (TP + FN) > 0 else 0.0

    # Specificity (True Negative Rate) = TN / (TN + FP)
    specificity[class_names[i]] = TN / (TN + FP) if (TN + FP) > 0 else 0.0

print("Sensitivity (True Positive Rate) per class:")
for class_name, sens in sensitivity.items():
    print(f"{class_name}: {sens:.4f}")

print("\nSpecificity (True Negative Rate) per class:")
for class_name, spec in specificity.items():
    print(f"{class_name}: {spec:.4f}")

In [ ]:
from sklearn.metrics import roc_curve, auc
from itertools import cycle

# Get true labels and predicted probabilities
y_true = np.concatenate([y.numpy() for x, y in test_dataset], axis=0)
y_pred_prob = model.predict(test_dataset)

# Binarize the true labels for ROC curve calculation
y_true_bin = tf.keras.utils.to_categorical(y_true, num_classes=len(class_names))

# Compute ROC curve and ROC area for each class
fpr = dict()
tpr = dict()
roc_auc = dict()
for i in range(len(class_names)):
    fpr[i], tpr[i], _ = roc_curve(y_true_bin[:, i], y_pred_prob[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])

# Plot ROC curves
plt.figure(figsize=(10, 8))
colors = cycle(['aqua', 'darkorange', 'cornflowerblue', 'deeppink', 'navy', 'green', 'red'])
for i, color in zip(range(len(class_names)), colors):
    plt.plot(fpr[i], tpr[i], color=color, lw=2,
             label='ROC curve of class {0} (area = {1:0.2f})'.format(class_names[i], roc_auc[i]))

plt.plot([0, 1], [0, 1], 'k--', lw=2)
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.legend(loc="lower right")
plt.show()

In [ ]:
!pip install gradio

import gradio as gr
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing import image

# ---------------------------
# Load your trained model
# ---------------------------
# model = tf.keras.models.load_model("/content/drive/.../your_model.h5")
# class_names = ['Atelectasis','Cardiomegaly','EmphReform','Hernia','Normal','Pneumonia','Tuberculosis']

def predict_cxr(img):
    # Preprocess image
    img = img.resize((224,224))
    img_array = tf.keras.preprocessing.image.img_to_array(img)/255.0
    img_array = np.expand_dims(img_array, axis=0)

    # Predict
    preds = model.predict(img_array)[0]
    preds_percent = np.round(preds * 100, 2)

    # Clinical notes dictionary
    clinical_notes = {
        "Atelectasis": "Collapsed lung tissue",
        "Cardiomegaly": "Enlarged heart silhouette",
        "EmphReform": "Emphysema / lung damage",
        "Hernia": "Lung herniation abnormality",
        "Normal": "No visible pathology",
        "Pneumonia": "Lung infection / consolidation",
        "Tuberculosis": "Bacterial lung infection"
    }

    # Monte Carlo dropout (uncertainty)
    # Do 5 forward passes
    mc_preds = []
    for _ in range(5):
        mc_preds.append(model.predict(img_array)[0])
    mc_preds = np.array(mc_preds)
    uncertainty = float(np.std(mc_preds))

    # Format output
    label_scores = {class_names[i]: float(preds_percent[i]) for i in range(len(class_names))}
    notes = "\n".join([f"{cls}: {clinical_notes[cls]}" for cls in class_names])

    return label_scores, f"Epistemic Uncertainty Score (MCD): {uncertainty:.6f}", notes


# ---------------------------
# Gradio UI Layout
# ---------------------------

with gr.Blocks(title="Chest X-Ray Abnormality Prediction") as demo:
    gr.Markdown("<h1>Chest X-Ray Abnormality Prediction</h1>")
    gr.Markdown("Upload a CXR image to predict disease probabilities & uncertainty")

    with gr.Row():
        with gr.Column():
            img_input = gr.Image(type="pil", label="Upload Chest X-Ray")
        with gr.Column():
            label_output = gr.Label(num_top_classes=7, label="Prediction Results")
            uncertainty_output = gr.Textbox(label="Uncertainty Score")
            clinical_output = gr.Textbox(label="Disease Info (Clinical Notes)")

    btn = gr.Button("Predict")
    btn.click(predict_cxr, inputs=img_input, outputs=[label_output, uncertainty_output, clinical_output])

demo.launch(debug=True)


In [ ]:
# Save the full model (architecture + weights + optimizer)
model.save("resnet50_cnn_chestxray_model")


In [ ]:
model.save("resnet50_cnn_chestxray_model.h5")


In [ ]:
from tensorflow.keras.models import load_model

model = load_model("resnet50_cnn_chestxray_model")


In [ ]:
model = load_model("resnet50_cnn_chestxray_model.h5")


In [ ]:
import pickle

with open("class_names.pkl", "wb") as f:
    pickle.dump(class_names, f)


In [ ]:
with open("class_names.pkl", "rb") as f:
    class_names = pickle.load(f)
